In [1]:
MODEL_NAME="gemma-4-26b-a4b-it"

In [3]:
!pip install -qU google-genai "pydantic>=2.7"

In [5]:
import os 
from getpass import getpass
from google import genai
from google.genai import types
print("Import Successful")

Import Successful


In [6]:
API_KEY=os.getenv("GEMINI_API_KEY")
if not API_KEY:
    API_KEY=getpass("Enter Gemini API Key: ")

client=genai.Client(api_key=API_KEY)

print("Gemini API client created successfully")

Enter Gemini API Key:  ········


Gemini API client created successfully


In [8]:
#Verify hosted Gemma model
available_gemma_models=[]

for model in client.models.list():
    model_name=model.name.removeprefix("models/")

    if "gemma" in model_name.lower():
        available_gemma_models.append(model_name)

print("Gemma models available to this API key: ")

for name in available_gemma_models:
    print(" -", name)

if not available_gemma_models:
    raise RuntimeError(
        "No hosted Gemma models were returned for this API key/project. "
        "Check the key, project, billing/quota configuration, and regional availability.")

Gemma models available to this API key: 
 - gemma-4-26b-a4b-it
 - gemma-4-31b-it


In [9]:
MODEL_NAME="gemma-4-26b-a4b-it"

if MODEL_NAME not in available_gemma_models:
    raise RuntimeError(
        f"{MODEL_NAME!r} is not available to this API key."
        f"Available Gemma models: {available_gemma_models}")

In [11]:
from typing import Literal
from pydantic import BaseModel, Field

class BoundaryValues(BaseModel):
    minimum:int
    middle:int
    maximum:int

class Variables(BaseModel):
    FICO: BoundaryValues
    NOINQ: BoundaryValues

class Combination(BaseModel):
    FICO:int
    NOINQ: int
    DECISIONCD: Literal["Declined"]
    DECISIONDESC: Literal["INQUIRIES"]

class ModelResult(BaseModel):
    variables: Variables
    combinations: list[Combination]=Field(min_length=9, max_length=9)

In [12]:
RULE="""IF FICO>750 AND FICO<=900 and NOINQ>=2 and NOINQ<=99 then DECISIONCD in Declines and DECISIONDESC is INQUIRIES""".strip()

In [14]:
SYSTEM_INSTRUCTION = """
You are a boundary-value test-case generator.

Follow the supplied algorithm exactly. Perform the calculations internally.
Your final response must contain the only one valid JSON object.
Do not output reasoning, Markdown, headings, comments, or code fences."""

In [26]:
USER_PROMPT=f""""
Generate boundary-value test cases for exactly this decision rule:

<RULE>
{RULE}
</RULE>

1. Treat FICO and NOINQ as integer-valued variables
2. For each variable, find its admissible inclusive minimum and maximum:
    -x>a means the inclusive minimum is a+1
    -x>=a means the inclusive minimum is a
    -x<b means the inclusive maximum is a-1
    -x<=b means the inclusive maximum is b
3. Compute the middle var as:
    floor((inclusive_minimum+inclusive_maximum)/2)
4. Each variable must therefore have exactly three states:
    minimum, middle, maximum
5. Generate the complete Cartesian product of those states.
6. There are two variables and three states per variable , so retun
  exactly 9 unique combinations.
7. Preserve the consequent values exactly:
    DECISIONCD="Declined"
    DECISIONDESC="INQUIRIES"

OUTPUT FORMAT
Return exactly this JSON structure:
{{
    "variables": {{
    "FICO":{{
            "minimum":<integer>
            "middle":<integer>
            "maximum":<integer>
            }},
    "NOINQ" {{
            "minimum":<integer>
            "middle":<integer>
            "maximum":<integer>
            }},
            }}
    "combinations":[
            {{
            "FICO":<integer>
            "NOINQ":<integer>
            "DECISIONCD":<integer>
            "DECISIONDESC":<integer>
            
    ]
        
All numeric values must be JSON integers.
Do not omit or duplicate any combination.
Do not include markdown fences, commentary, reasoning, headings , or extra keys."""


In [27]:
response=client.models.generate_content(
    model=MODEL_NAME,
    contents=USER_PROMPT,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0,
        max_output_tokens=2048,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    ),
)



In [28]:
raw_response=response.text.strip()
print(raw_response)

{
"variables": {
"FICO": {
"minimum": 751,
"middle": 825,
"maximum": 900
},
"NOINQ": {
"minimum": 2,
"middle": 50,
"maximum": 99
}
},
"combinations": [
{
"FICO": 751,
"NOINQ": 2,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 751,
"NOINQ": 50,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 751,
"NOINQ": 99,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 825,
"NOINQ": 2,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 825,
"NOINQ": 50,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 825,
"NOINQ": 99,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 900,
"NOINQ": 2,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 900,
"NOINQ": 50,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
},
{
"FICO": 900,
"NOINQ": 99,
"DECISIONCD": "Declined",
"DECISIONDESC": "INQUIRIES"
}
]
}


In [29]:
import json

In [30]:
def extract_json_object(text: str) -> dict:
    """
    Extract the first complete JSON object from an LLM response.
    This parses the response, not the decision rule.
    """
    start=text.find("{")

    if start==-1:
        raise ValueError("The model response did not contain a JSON object.")

    decoder=json.JSONDecoder()

    try:
        payload, end_position=decoder.raw_decode(text[start:])
    except json.JSONDecoderError as exc:
        raise ValueError(
            f"The model returned invalid JSON: {exc}"
        ) from exc

    if not isinstance(payload, dict):
        raise TypeError("The top-level model response must be a JSON object.")
    return payload

json_payload=extract_json_object(raw_response)
parsed_result=ModelResult.model_validate(json_payload)

print("Response JSON and schema validation passed")

Response JSON and schema validation passed
